<a href="https://colab.research.google.com/github/fboliveira/Evolutionary-Computation/blob/main/Codes/TestFunction/ga_binary_refactored.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bibliotecas utilizadas
import random

In [ ]:
# GA - Representação binária
# Precisão das variáveis - quantos bits para cada variável
precisao = 4

# Número de variáveis
variaveis = 5

# Valores mínimo e máximo
min = -10
max = 10

# Tamanho do cromossomo - número de variáveis * quantidade de bits por variável
tamanho_cromossomo = variaveis * precisao

# Ranking linear:
# Definir a pressão seletiva -> SP[1.0, 2.0]
SP = 1.2

# Parâmetros do GA:

# Quantos indivíduos serão utilziados - tamanho da populacao
tamanho_populacao = 10

# Critério de parada número máximo de gerações
geracoes = 10

# Taxa de crossover - operador principal do GA
taxa_crossover = 0.8

# Taxa de mutação - operador secundário do GA
taxa_mutacao = 0.05

In [ ]:
# Avaliação da população inicial
# - decodificar de binário para inteiro
# - transformar de inteiro para real, considerando o intervalo das variáveis [min, max]
def calcular_funcao_objetivo(individuo):

  # Decodificar de binário para inteiro
  lista_de_inteiros = []

  # Para cada variável:
  for i in range(variaveis):
    # Para cada um dos bits na precisão
    valor_inteiro = 0
    for bit in range(precisao):
      # 2^indice
      #  6543210
      # [0000000 0010101 0010101 0010101 1111111]
      #                    b
      # 2^indice * valor do bit
      valor_inteiro += pow(2, precisao - 1 - bit) * individuo[i * precisao + bit]
    # ----
    # Adicionar o valor inteiro da variável na lista de inteiros
    lista_de_inteiros.append(valor_inteiro)

  # Transformar os números inteiros (lista) para o valor real
  # Mínimo e o máximo => [min, max]
  valores_reais = []

  # Para cada inteiro da lista de inteiros, transformar
  # Para cada valor inteiro transformado
  for inteiro in lista_de_inteiros:
    real = ( inteiro * ( ( max - min ) / ( pow(2, precisao) - 1 ) ) ) + min
    valores_reais.append(real)

  # Com os valores reais, calcular o custo da função objetivo
  # sum(x_i^2)
  custo = 0

  for valor in valores_reais:
    custo += valor ** 2

  return custo

In [ ]:
# Geração da população inicial - clássico: aleatória

# Lista (vetor/array) para representar o indivíduo: individuo = []
# População - conjunto de listas (listas de listas)
populacao = []

# Para cada indívuduo da população
# Gerar um valor aleatório (bit - 0 ou 1) até o tamanho do indivíduo
# range: [start, stop[ : 1-n => 1 até (n - 1)
for i in range(tamanho_populacao):
  # Para cada variável:
    # Para cada valor da precisao:
  individuo = []
  # Para cada valor do indivíduo - tamanho do cromossomo
  for valor in range(tamanho_cromossomo):
    bit = random.randint(0, 1)
    individuo.append(bit)

  custo = calcular_funcao_objetivo(individuo)

  # Adicionar o indivíduo na população inicial
  populacao.append((individuo, custo))

In [ ]:
# Para cada individuo da populacao, avaliar o custo:
# Desncessário (ou redundante): o indivíduo tem o custo associado -> tupla (indivíduo, custo)
custos_funcao = []

# tupla -> [0]: indivíduo; [1]: custo (função objetivo)

for tupla in populacao:
  # Calcular o custo do indivíduo
  custo = calcular_funcao_objetivo(tupla[0])
  # Adicionar o custo à lista de funções de custo
  custos_funcao.append(custo)

In [ ]:
# Avaliação da população - ranking linear
# Ordenar a população do pior para o melhor

def avaliar_populacao(populacao):

  # Classificar a população (ordem decrescente): tupla[1] -> custo
  populacao.sort(key = lambda tupla : tupla[1], reverse=True)
  # print(populacao)

  fitness = []

  # fitness(i) = 2 - SP + 2 * (SP - 1) * ( ( i - 1 ) / ( nInd - 1 ) )
  for i in range(1, tamanho_populacao + 1):
    fitness_individuo = 2 - SP + 2 * ( SP - 1 ) * ( (i - 1) / ( tamanho_populacao - 1 ) )
    fitness.append(fitness_individuo)

  return fitness

In [ ]:
# Estrutura de repetição
# Custo da melhor solução - geração inicial: último indivíduo -> tupla[1]
fitness = avaliar_populacao(populacao)
melhor_custo_ate_agora = populacao[ tamanho_populacao - 1 ][1]

print('Custo da solução inicial: ', melhor_custo_ate_agora)

# Iterar sobre essas gerações - processo de evolução/melhoria da solução
for geracao in range(geracoes):

  # Melhor custo até agora
  melhor_custo_ate_agora = populacao[ tamanho_populacao - 1 ][1]
  print(f'Geração: {geracao} - custo: {melhor_custo_ate_agora}')

  # Seleção dos progenitores:
  # - Aleatória ?
  # - Seleção por torneio? Dois a dois

  # - Seleção por roleta - fitness
  proporcao_fitness = []
  # Exemplo slide - roleto
  # [ 14.4; 49.2;  5.5;  30.9 ]
  # [ 14.4; 63.6; 69.1; 100.0 ]
  soma_fitness = sum(fitness)
  soma_proporcao = 0.0
  for valor in fitness:
    # soma_proporcao = soma_proporcao + (valor / soma_fitness)
    soma_proporcao += (valor / soma_fitness)
    proporcao_fitness.append(soma_proporcao)

  # Descendentes
  descendentes = []

  # Fator de reprodução
  # Taxa de descendentes - taxa de crossover
  numero_descendentes = round( taxa_crossover * tamanho_populacao )

  # Processo de reprodução:
  for d in range(numero_descendentes):

    # Selecionar os progenitores a partir do fitness - critério método da roleta

    # - Selecionar o índice do primeiro progenitor:
    rnd_primeiro_progenitor = random.random() # 50%

    indice_primeiro_progenitor = 0

    for indice, valor in enumerate(proporcao_fitness):
      if ( rnd_primeiro_progenitor <= valor ):
        indice_primeiro_progenitor = indice # 1
        break

    # - Selecionar o índice do segundo progenitor:
    # -- Garantir índices diferentes para os progenitores
    indice_segundo_progenitor = indice_primeiro_progenitor

    while ( indice_segundo_progenitor == indice_primeiro_progenitor ):

      rnd_segundo_progenitor = random.random() # 80%

      for indice, valor in enumerate(proporcao_fitness):
        if ( rnd_segundo_progenitor <= valor ):
          indice_segundo_progenitor = indice # 3
          break

    # - Recuperar da população os progenitores - tupla[0]
    primeiro_progenitor = populacao[ indice_primeiro_progenitor ][0]
    segundo_progenitor = populacao[ indice_segundo_progenitor ][0]

    # - Operação de crossover
    # -- Pontos de corte: representação binária - ordem não importa
    ponto_de_corte = random.randint(0, tamanho_populacao - 1)

    # -- Primeiro descendente
    # D1 = P1_1 + P2_2
    primeiro_descendente = primeiro_progenitor[0:ponto_de_corte] + segundo_progenitor[ponto_de_corte:len(segundo_progenitor)]

    # -- Segundo descendente
    # D2 = P2_1 + P1_2
    segundo_descendente = segundo_progenitor[0:ponto_de_corte] + primeiro_progenitor[ponto_de_corte:len(primeiro_progenitor)]

    # Mutação - operador secundário
    # - Ocorre nos descendentes após (em seguida) ao processo de crossover

    # -- Possibilidade de mutação em UM dos bits (apenas)
    probabilidade_mutacao = random.random()

    if (probabilidade_mutacao <= taxa_mutacao):
      # Aplicar a mutação em um dos bits - numa posição aleatória
      ponto_de_mutacao = random.randint(0, len(primeiro_descendente) - 1)
      # Inverter o valor do bit
      if (primeiro_descendente[ ponto_de_mutacao ] == 0):
        primeiro_descendente[ponto_de_mutacao] = 1
      else:
        primeiro_descendente[ponto_de_mutacao] = 0

    # -- Possibilidade de mutação por bit - para cada uma das posições
    for posicao in range(len(segundo_descendente)):
      probabilidade_mutacao = random.random()
      ponto_de_mutacao = posicao

      if ( probabilidade_mutacao <= taxa_mutacao ):
        # Inverter o valor do bit
        if (segundo_descendente[ponto_de_mutacao] == 0):
          segundo_descendente[ponto_de_mutacao] = 1
        else:
          segundo_descendente[ponto_de_mutacao] = 0

    # -- Encerra o processo de reprodução para os dois desecendentes

    # Calcular o custo dos descendentes
    custo_primeiro_descendente = calcular_funcao_objetivo(primeiro_descendente)
    custo_segundo_descendente = calcular_funcao_objetivo(segundo_descendente)

    # GA Memético -> GA com busca local
    # -- processo de busca de local, considerando uma taxa de probabalilidade para cada um deles.
    # -- TSP/VRP, problema combinatório: representação inteira.

    # Adicionar os descendentes na lista -> definir os sobreviventes
    descendentes.append( (primeiro_descendente, custo_primeiro_descendente) )
    descendentes.append( (segundo_descendente, custo_segundo_descendente) )

    # -- Encerra o for do processo de reprodução para população

  # Avaliação da população: a população atual e os descedentes
  # -- Combinar as duas populações: atual + descedentes (merge)
  nova_populacao = populacao + descendentes

  # -- Ranking linear da nova populacao [pior para o melhor]: roleta
  # fitness_nova_populacao = avaliar_populacao(nova_populacao)
  nova_populacao.sort(key = lambda tupla : tupla[1]) # asc [melhor para o pior]

  # Definir os sobreviventes:
  # -- Fitness e seleção por roleta (bons e ruins) -> não se tem garantir que o melhor indivíduo será selecionado
  # -- Parte elitismo (5%), parte fitness/roleta (75%), parte torneio (20%)
  # -- Substituir parte da população por novos indivíduos (10% aleatória) -> garantir diversidade na população
  # -- Selecionar os n primeiros melhores (tamanho_populacao) -> 100% elitismo
  populacao = nova_populacao[0:tamanho_populacao]

  # Avaliar a população sobrevivente
  fitness = avaliar_populacao(populacao) # [do pior para o melhor]


Custo da solução inicial:  51.99999999999997
Geração: 0 - custo: 51.99999999999997
Geração: 1 - custo: 51.99999999999997
Geração: 2 - custo: 51.99999999999997
Geração: 3 - custo: 51.99999999999997
Geração: 4 - custo: 51.99999999999997
Geração: 5 - custo: 41.333333333333314
Geração: 6 - custo: 41.333333333333314
Geração: 7 - custo: 27.111111111111104
Geração: 8 - custo: 23.55555555555555
Geração: 9 - custo: 19.999999999999993


In [ ]:
# Lista o melhor indivíduo
print(populacao)

# Melhor resultado:
melhor_individuo = populacao[ tamanho_populacao - 1 ][0]
melhor_custo = populacao[ tamanho_populacao - 1 ][1]

print("Melhor custo: ", melhor_custo)
print("Melhor individuo: ", melhor_individuo)

[([1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 27.111111111111104), ([1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 23.55555555555555), ([1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 23.55555555555555), ([1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 23.55555555555555), ([1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 23.55555555555555), ([1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 23.55555555555555), ([1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 23.55555555555555), ([1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 23.555555555555546), ([1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 19.999999999999993), ([1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1], 19.999999999999993)]
Melhor custo:  19.999999999999993
Melhor individuo:  [1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1]
